### **Laboratorio 9. Recuperación, control y evaluación de agentes con datos reales**

**Modalidad:** 4 horas. No se aceptan resultados inventados, consultas artificiales ni muestreos favorables. El laboratorio trabaja con el **SQuAD dev completo** y separa resultados offline de Qwen real.

**Entrega única:** notebook ejecutado con métricas calculadas en tu Docker, IDs de fallos, explicación y conclusiones de alcance delimitado.

#### **1. Confirmación del dataset y particiones originales**

In [ ]:
from pathlib import Path
import csv, json, sys
root = Path.cwd().resolve()
for parent in (root, *root.parents):
    if (parent / "Semana9" / "agent_benchmark.py").is_file():
        root=parent
        break
else:
    raise FileNotFoundError("Inicia Jupyter desde CC-0F4")
sys.path.insert(0,str(root))
from Semana9.prepare_real_data import verify
from Semana9.agent_benchmark import BM25Tool, jsonl, real_paths, evaluate_run, lexical_coverage
m=verify(root/"Semana9"/"datos_reales")
print("Fuente verificable SHA256:",m["source_sha256"])
print("Cantidades originales:",m["articles"],m["passages"],m["queries"])
print("Preguntas por partición:",m["split_queries"])
assert m["queries"]==10570 and m["passages"]==2067

**Actividad 1 (razonamiento).** Identifica cómo se eligieron los artículos de calibración, por qué no se pueden mezclar con evaluación y por qué una pregunta humana no implica automáticamente que su recuperación sea difícil.

#### **2. Ejecutar A0, A1 y B sin usar las respuestas humanas en la política**

In [ ]:
summary=evaluate_run(root,root/"Semana9"/"resultados")
assert summary["threshold_origin"]=="calibration"
assert summary["calibration_queries"]+summary["evaluation_queries"]==10570
print("Umbral seleccionado solo en calibración:",summary["selected_threshold"])
print("Tasas en EVALUACIÓN:")
for label, item in summary["evaluation_results"].items():
    print(label, item)
print("Recall y MRR:",summary["retrieval_evaluation"])

**Actividad 2 (método).** Explica si el umbral seleccionado cumple la tolerancia de pérdida en calibración y si mantiene esa propiedad en evaluación. Anota el nombre del experimento y la cantidad real de consultas, sin redondear los resultados para que parezcan perfectos.

#### **3. Analizar fallos auténticos, no ejemplos previamente elegidos**

In [ ]:
with (root/"Semana9"/"resultados"/"runs.csv").open(encoding="utf-8",newline="") as handle:
    rows=list(csv.DictReader(handle))
questions={q["query_id"]:q for q in jsonl(root/"Semana9"/"datos_reales"/"queries.jsonl")}
corpus=jsonl(root/"Semana9"/"datos_reales"/"corpus.jsonl")
idx={p["passage_id"]:p["text"] for doc in corpus for p in doc["passages"]}
failed=[r for r in rows if r["split"]=="evaluation" and r["policy"]=="workflow_fixed_top3" and r["evidence_hit"]=="0"]
print("Fallos evaluados top3:",len(failed))
for r in failed[:5]:
    q=questions[r["query_id"]]
    print("CASO:",q["query_id"],q["query"])
    print("Pasaje de origen:",idx[q["passage_id"]][:280])
    print("Top3:",r["citations"],"| cobertura:",r["top1_coverage"])

**Actividad 3 (análisis de error).** Selecciona **dos errores observados**, anota sus IDs y explica si parecen deberse a vocabulario, ambigüedad, cambios de formulación o falta de información. La categoría debe justificarse con texto original, no asignarse por la puntuación solamente.

#### **4. Economía del contexto frente a recuperación**

In [ ]:
from collections import defaultdict
paired=defaultdict(dict)
for r in rows:
    if r["split"]=="evaluation":paired[r["query_id"]][r["policy"]]=r
violations=[]; saved=[]; extra=[]
for qid, d in paired.items():
    a0=d["workflow_fixed_top1"];a1=d["workflow_fixed_top3"];b=d["adaptive_rule_based"]
    if not (int(a0["evidence_hit"])<=int(b["evidence_hit"])<=int(a1["evidence_hit"])):
        violations.append(qid)
    saved.append(int(a1["context_word_proxy"])-int(b["context_word_proxy"]))
    extra.append(int(b["tool_calls"])-int(a1["tool_calls"]))
assert not violations
print("Consultas pareadas:",len(paired))
print("Ahorro promedio de palabras frente A1:",sum(saved)/len(saved))
print("Búsquedas adicionales promedio frente A1:",sum(extra)/len(extra))

**Actividad 4 (decisión de ingeniería).** ¿Compensa el ahorro de contexto el número adicional de búsquedas? Justifica con **métricas de evaluación**, no solo con teoría. ¿Puede B superar a A1 en recuperación con la implementación actual? ¿Qué cambiaría para que la respuesta fuera sí?

#### **5. Contratos y comprobaciones sin ejecutar acciones peligrosas**

In [ ]:
import unittest
suite=unittest.TestSuite()
for name in ("Semana9.test_agent_loop","Semana9.test_agent_benchmark","Semana9.test_prepare_real_data","Semana9.test_agent_llm_eval"):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromName(name))
tests=unittest.TextTestRunner(verbosity=1).run(suite)
assert tests.wasSuccessful()

**Actividad 5 (seguridad).** Elige una prueba de `test_agent_loop.py` sobre cita no observada o herramienta no autorizada. Explica por qué el controlador la bloquea incluso si un modelo solicita esa acción. **No confundas los casos de prueba con el dataset experimental SQuAD.**

#### **6. Solo si se ejecutó Qwen: comparación pareada con el mismo modelo**

In [ ]:
llm_summary=root/"Semana9"/"resultados"/"llm_real"/"resumen_llm.json"
if not llm_summary.exists():
    print("EVALUACIÓN QWEN NO EJECUTADA. No inventar EM, F1 ni citas de agente.")
else:
    result=json.loads(llm_summary.read_text(encoding="utf-8"))
    print("Modelo:",result["model_id"],"| consultas:",result["queries"])
    print(json.dumps(result["metrics"],ensure_ascii=False,indent=2))

**Actividad 6 (opcional, solo con inferencia real).** Compara EM/F1 del workflow fijo con el agente usando el mismo Qwen y el mismo subconjunto de preguntas. Identifica un fallo de respuesta incorrecta con cita válida y explica qué validación adicional se necesita para afirmar *grounding*. Si no ejecutaste Qwen, declara `NO EJECUTADO`.

#### **Entrega y defensa breve**

Incluye el notebook con resultados reales, dos IDs de fallos auténticos, una explicación de la política adaptativa, la salida de los tests de contrato y una conclusión que distinga recuperación, seguridad de herramientas y corrección de respuestas. No edites consultas, pasajes ni respuestas originales después de inspeccionar resultados.